<a href="https://colab.research.google.com/github/Simranjit15kaur/Kth_worldModel/blob/main/Colab2_ConvLSTM_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab 2 — ConvLSTM Dynamics Training
**KTH World Model Pipeline**

**Requires:** `vae_v2.pth` uploaded by Colab 1

```
Colab 1 → trains VAE         → uploads vae_v2.pth
Colab 2 → downloads VAE      → trains ConvLSTM → uploads dynamics_v2.pth
Colab 3 → downloads VAE      → trains Spatial TX (run in parallel with Colab 2)
Colab 4 → downloads all 3    → evaluation + GIFs
```

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CONFIG — same values as Colab 1
# ══════════════════════════════════════════════════════════════════════════
HF_TOKEN   = #entered
HF_REPO_ID = #entered

# Training config
# Tip: set DYNAMICS_EPOCHS=100 for better results (run overnight)
DYNAMICS_EPOCHS = 100     # 50 minimum, 100 recommended
BATCH_SIZE      = 8
DATA_PATH       = '/content/kth_data'
SAVE_PATH       = '/content'
KTH_ACTIONS     = ['walking', 'jogging']

In [ ]:
!pip install torch torchvision scikit-image imageio tqdm lpips Pillow huggingface_hub -q

In [ ]:
import os, glob, random, math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import numpy as np
from PIL import Image
from tqdm import tqdm
import lpips

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

os.makedirs(SAVE_PATH, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)

lpips_fn = lpips.LPIPS(net='alex').to(device)
for p in lpips_fn.parameters(): p.requires_grad = False

In [ ]:
# ── Download KTH data ─────────────────────────────────────────────────────
KTH_BASE_URL = 'https://www.csc.kth.se/cvap/actions/'

def download_kth_lean(actions, data_path):
    import requests, zipfile
    for action in actions:
        zip_path     = os.path.join(data_path, f'{action}.zip')
        extract_path = os.path.join(data_path, action)
        if os.path.exists(extract_path) and len(os.listdir(extract_path)) > 0:
            print(f'  {action} already exists — skipping'); continue
        print(f'  Downloading {action}...')
        with requests.get(KTH_BASE_URL + f'{action}.zip', stream=True) as r:
            r.raise_for_status()
            with open(zip_path, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192): f.write(chunk)
        with zipfile.ZipFile(zip_path, 'r') as z: z.extractall(data_path)
        os.remove(zip_path)
        print(f'  {action} done.')

download_kth_lean(KTH_ACTIONS, DATA_PATH)

In [ ]:
# ── KTH Dataset (identical to Colab 1) ───────────────────────────────────
class KTHDataset(Dataset):
    IMG_SIZE = 64
    SEQ_LEN  = 20

    def __init__(self, data_path, actions, split='train', augment=True):
        self.augment   = augment
        self.sequences = []
        self.transform = T.Compose([
            T.Grayscale(),
            T.Resize((self.IMG_SIZE, self.IMG_SIZE),
                     interpolation=T.InterpolationMode.LANCZOS, antialias=True),
            T.ToTensor(),
        ])
        train_persons = {f'person{i:02d}' for i in range(1, 17)}
        test_persons  = {f'person{i:02d}' for i in range(17, 26)}
        allowed = train_persons if split == 'train' else test_persons
        all_videos = glob.glob(os.path.join(data_path, '*.avi'))
        print(f'Found {len(all_videos)} total videos')
        for vid_file in sorted(all_videos):
            filename = os.path.basename(vid_file).lower()
            if not any(action in filename for action in actions): continue
            person = filename.split('_')[0]
            if person not in allowed: continue
            frames = self._load_video(vid_file)
            for start in range(0, len(frames) - self.SEQ_LEN, self.SEQ_LEN // 2):
                clip = frames[start:start + self.SEQ_LEN]
                if len(clip) == self.SEQ_LEN: self.sequences.append(clip)
        print(f'KTH {split}: {len(self.sequences)} sequences')

    def _load_video(self, path):
        import cv2
        cap = cv2.VideoCapture(path)
        frames = []
        while True:
            ret, frame = cap.read()
            if not ret: break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            t = self.transform(Image.fromarray(frame))
            t_min, t_max = t.min(), t.max()
            if t_max > t_min: t = (t - t_min) / (t_max - t_min)
            frames.append(t)
        cap.release()
        return frames

    def __len__(self): return len(self.sequences)

    def __getitem__(self, idx):
        seq = torch.stack(self.sequences[idx], dim=0)
        if self.augment and random.random() > 0.5:
            seq = torch.flip(seq, dims=[3])
        return seq


train_dataset = KTHDataset(DATA_PATH, KTH_ACTIONS, split='train', augment=True)
test_dataset  = KTHDataset(DATA_PATH, KTH_ACTIONS, split='test',  augment=False)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=2, pin_memory=True)
print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

In [ ]:
# ── VAE definition (needed to load weights and encode sequences) ──────────
class VAE(nn.Module):
    def __init__(self, latent_channels=128):
        super().__init__()
        self.latent_channels = latent_channels
        self.encoder = nn.Sequential(
            nn.Conv2d(1,   32,  4, stride=2, padding=1), nn.LeakyReLU(0.2),
            nn.Conv2d(32,  64,  4, stride=2, padding=1), nn.LeakyReLU(0.2),
            nn.Conv2d(64,  128, 4, stride=2, padding=1), nn.LeakyReLU(0.2),
            nn.Conv2d(128, latent_channels * 2, 3, stride=1, padding=1),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(latent_channels, 128, 3, stride=1, padding=1), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(64,  32, 4, stride=2, padding=1), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(32,   1, 4, stride=2, padding=1), nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x)
        mu, logvar = h.chunk(2, dim=1)
        return mu, torch.clamp(logvar, -10, 10)

    def reparameterize(self, mu, logvar):
        return mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)

    def decode(self, z): return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        return self.decode(self.reparameterize(mu, logvar)), mu, logvar


# ── Download VAE from Hugging Face ────────────────────────────────────────
from huggingface_hub import hf_hub_download

vae_ckpt_path = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename='vae_v2.pth',
    token=HF_TOKEN
)
print(f'Downloaded: {vae_ckpt_path}')

vae_v2 = VAE().to(device)
ckpt   = torch.load(vae_ckpt_path, map_location=device)
vae_v2.load_state_dict(ckpt['model'])
vae_v2.eval()
for p in vae_v2.parameters(): p.requires_grad = False
print(f'VAE v2 loaded and frozen | Training loss was: {ckpt["losses"][-1]:.4f}')

In [ ]:
# ── ConvLSTM ──────────────────────────────────────────────────────────────
class ConvLSTMCell(nn.Module):
    def __init__(self, in_channels, hidden_channels, kernel_size=3):
        super().__init__()
        pad = kernel_size // 2
        self.hidden_channels = hidden_channels
        self.conv = nn.Conv2d(in_channels + hidden_channels,
                              4 * hidden_channels, kernel_size, padding=pad)

    def forward(self, x, h, c):
        i, f, o, g = self.conv(torch.cat([x, h], dim=1)).chunk(4, dim=1)
        c_next = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(g)
        return torch.sigmoid(o) * torch.tanh(c_next), c_next

    def init_hidden(self, batch_size, spatial_size, device):
        H, W = spatial_size
        return (torch.zeros(batch_size, self.hidden_channels, H, W, device=device),
                torch.zeros(batch_size, self.hidden_channels, H, W, device=device))


class LatentDynamicsModel(nn.Module):
    def __init__(self, latent_channels=128):
        super().__init__()
        self.cell1 = ConvLSTMCell(latent_channels, latent_channels)
        self.cell2 = ConvLSTMCell(latent_channels, latent_channels)

    def forward(self, latent_seq):
        B, T, C, H, W = latent_seq.shape
        dev = latent_seq.device
        h1, c1 = self.cell1.init_hidden(B, (H, W), dev)
        h2, c2 = self.cell2.init_hidden(B, (H, W), dev)
        outputs = []
        for t in range(T):
            h1, c1 = self.cell1(latent_seq[:, t], h1, c1)
            h2, c2 = self.cell2(h1, h2, c2)
            outputs.append(h2)
        return torch.stack(outputs, dim=1)


def encode_sequence(vae, frames_seq):
    B, T = frames_seq.shape[:2]
    return torch.stack([vae.encode(frames_seq[:, t])[0] for t in range(T)], dim=1)


dynamics_model = LatentDynamicsModel().to(device)
print(f'ConvLSTM | {sum(p.numel() for p in dynamics_model.parameters())/1e6:.2f}M params')

In [ ]:
# ── Train ConvLSTM with resume support ────────────────────────────────────
DYNAMICS_CKPT = f'{SAVE_PATH}/dynamics_v2_ckpt.pth'
DYNAMICS_FINAL = f'{SAVE_PATH}/dynamics_v2.pth'
dynamics_losses = []
start_epoch     = 0

dynamics_optimizer = optim.Adam(dynamics_model.parameters(), lr=1e-3)
dynamics_scheduler = optim.lr_scheduler.CosineAnnealingLR(
    dynamics_optimizer, T_max=DYNAMICS_EPOCHS)

# Check local checkpoint first, then try HF Hub
if os.path.exists(DYNAMICS_CKPT):
    ckpt = torch.load(DYNAMICS_CKPT, map_location=device)
    dynamics_model.load_state_dict(ckpt['model'])
    dynamics_optimizer.load_state_dict(ckpt['optimizer'])
    dynamics_scheduler.load_state_dict(ckpt['scheduler'])
    dynamics_losses = ckpt['losses']
    start_epoch     = ckpt['epoch'] + 1
    print(f'Resumed from local checkpoint epoch {start_epoch}')
else:
    # Try downloading partial checkpoint from HF Hub (if previous run uploaded one)
    try:
        from huggingface_hub import hf_hub_download
        dl_path = hf_hub_download(HF_REPO_ID, 'dynamics_v2.pth', token=HF_TOKEN)
        ckpt = torch.load(dl_path, map_location=device)
        dynamics_model.load_state_dict(ckpt['model'])
        dynamics_losses = ckpt['losses']
        start_epoch     = len(dynamics_losses)
        for _ in range(start_epoch): dynamics_scheduler.step()
        print(f'Resumed from HF Hub — epoch {start_epoch}')
    except Exception:
        print('Starting fresh ConvLSTM training')

dynamics_model.train()
for epoch in range(start_epoch, DYNAMICS_EPOCHS):
    # epsilon ramp: 0.0 → 0.5 over first half, then hold at 0.5
    epsilon = min(0.5, (epoch / DYNAMICS_EPOCHS) * 0.5)
    total   = 0

    for batch in tqdm(train_loader, desc=f'ConvLSTM Epoch {epoch+1}/{DYNAMICS_EPOCHS} (ε={epsilon:.2f})'):
        batch = batch.to(device)
        with torch.no_grad():
            lat_in  = encode_sequence(vae_v2, batch[:, :10])
            lat_tgt = encode_sequence(vae_v2, batch[:, 10:])

        B, T, C, H, W = lat_in.shape
        dev = lat_in.device
        h1, c1 = dynamics_model.cell1.init_hidden(B, (H, W), dev)
        h2, c2 = dynamics_model.cell2.init_hidden(B, (H, W), dev)

        preds = []
        for t in range(T):
            if t == 0:
                inp = lat_in[:, 0]
            else:
                use_pred = torch.bernoulli(torch.ones(B, device=dev) * epsilon).bool()
                inp = torch.where(use_pred.view(B,1,1,1).expand_as(lat_in[:,t]),
                                  preds[-1].detach(), lat_in[:, t])
            h1, c1 = dynamics_model.cell1(inp, h1, c1)
            h2, c2 = dynamics_model.cell2(h1, h2, c2)
            preds.append(h2)

        loss = F.mse_loss(torch.stack(preds, dim=1), lat_tgt)
        dynamics_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(dynamics_model.parameters(), 1.0)
        dynamics_optimizer.step()
        total += loss.item()

    dynamics_scheduler.step()
    avg = total / len(train_loader)
    dynamics_losses.append(avg)

    if (epoch + 1) % 5 == 0:
        print(f'  Epoch {epoch+1:02d}/{DYNAMICS_EPOCHS}  Loss: {avg:.6f}')
        torch.save({'model': dynamics_model.state_dict(),
                    'optimizer': dynamics_optimizer.state_dict(),
                    'scheduler': dynamics_scheduler.state_dict(),
                    'losses': dynamics_losses,
                    'epoch': epoch}, DYNAMICS_CKPT)

# Save final
torch.save({'model': dynamics_model.state_dict(), 'losses': dynamics_losses}, DYNAMICS_FINAL)
if os.path.exists(DYNAMICS_CKPT): os.remove(DYNAMICS_CKPT)
print(f'ConvLSTM training complete. Final loss: {dynamics_losses[-1]:.6f}')

In [ ]:
# ── Upload to Hugging Face Hub ────────────────────────────────────────────
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=DYNAMICS_FINAL,
    path_in_repo='dynamics_v2.pth',
    repo_id=HF_REPO_ID,
    repo_type='model',
)
print(f'dynamics_v2.pth uploaded to https://huggingface.co/SimranjitKaur/vae_kth-world-model')
print(' Colab 4 can now download it.')

## Colab 2 Complete

ConvLSTM is trained and uploaded. Run Colab 3 (Spatial TX) if not already running. Then run Colab 4 for evaluation and GIF generation.